# Synthetic shapes

Runs the synthetic experiment step by step, saves every run under `results_synthetic/`, lists the per-run
numbers and draws the figure of calibrated 90% regions against the true region.

Each step calls `synthetic.py`, so a run here is the same as a run from the command line. Finished runs are
skipped, so a cell can be re-run after an interruption.

* `icnn` needs a GPU; the other methods run on the CPU with 8 threads.
* MVCS is by far the slowest method (about 1.5 hours per shape and seed on 8 cores).

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

import pandas as pd

REPO = Path.cwd() if (Path.cwd() / "synthetic.py").exists() else Path.cwd().parent
OUT = REPO / "results_synthetic"
sys.path.insert(0, str(REPO))


def run(shape, method, seeds):
    """python synthetic.py for the seeds that have no result yet, printing its output here."""
    todo = [s for s in seeds if not (OUT / shape / method / f"seed{s}" / "row.json").exists()]
    if not todo:
        print(f"{shape} / {method}: done")
        return
    cmd = [sys.executable, "-u", "synthetic.py", "--shape", shape, "--method", method,
           "--output-dir", str(OUT), "--seeds", *map(str, todo)]
    env = dict(os.environ)
    if method != "icnn":
        env["CUDA_VISIBLE_DEVICES"] = ""
    print("$ python", " ".join(cmd[2:]))
    with subprocess.Popen(cmd, cwd=REPO, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as p:
        for line in p.stdout:
            print(line, end="")
    if p.returncode:
        raise RuntimeError(f"{shape} / {method} failed")

## Settings

`SEEDS` are the seeds of the experiment; `FIGURE_SEEDS` picks one seed per shape for the figure. To only
draw the figure, set `SEEDS = None`: then just the figure seeds are run.

In [ ]:
SHAPES = ["ellipse", "rectangle", "triangle", "trapezoid"]
CPU_METHODS = ["absolute_residual", "empirical_copula", "global_ellipsoid", "local_ellipsoid", "spline", "mvcs"]
SEEDS = [42, 43, 44, 45, 46]
FIGURE_SEEDS = {"ellipse": 43, "rectangle": 43, "triangle": 43, "trapezoid": 44}


def seeds_for(shape):
    return SEEDS if SEEDS is not None else [FIGURE_SEEDS[shape]]

## 1. CPU methods

The spline is tuned first on each (shape, seed); the choice is saved in `results_synthetic/tuning/`.

In [ ]:
for shape in SHAPES:
    for method in CPU_METHODS:
        run(shape, method, seeds_for(shape))

## 2. CoCoNet<sub>ICNN</sub> (GPU)

In [ ]:
for shape in SHAPES:
    run(shape, "icnn", seeds_for(shape))

## 3. Results of every run

Test coverage, V/V\* (mean region area over the area of the true region) and IoU with the true region.

In [ ]:
rows = [json.loads(f.read_text()) for f in sorted(OUT.glob("*/*/seed*/row.json"))]
runs = pd.DataFrame(rows)[["shape", "method", "seed", "coverage", "v_over_v_star", "iou"]]
runs.sort_values(["shape", "method", "seed"]).reset_index(drop=True).round(4)

## 4. Figure

The calibrated 90% region of each method at x = (0, 0) for the figure seeds, over the test residuals and the
true region. Saved as `results_synthetic/synthetic_regions.pdf`.

In [ ]:
%matplotlib inline
from plot_synthetic import plot

fig = plot(OUT, [FIGURE_SEEDS[s] for s in SHAPES])
fig.savefig(OUT / "synthetic_regions.pdf", bbox_inches="tight")